In [ ]:
# ── Cell 1: load results + the two flip probabilities ───────────────────────
# Edge-flip under PURE DP vs APPROXIMATE DP at matched epsilon.
#   pure   : release truthfully w.p. p       = e^eps / (1 + e^eps)
#            -> flip w.p.  f       = 1 / (1 + e^eps)
#   approx : release truthfully w.p. p_delta = (e^eps + delta) / (1 + e^eps)
#            -> flip w.p.  f_delta = (1 - delta) / (1 + e^eps) = (1 - delta) * f
# Both arms see the same graph per rep; their flip randomness is independent
# (each arm is a separate application of the mechanism).
import glob

import numpy as np
import pandas as pd
from scipy.special import expit

from plot_style import method_color, method_label

OUTPUT_CSV = "./results/results_edgeflip_pure_vs_approx_dp.csv"   # match script_edgeflip_pure_vs_approx_dp.py

shard_files = sorted(glob.glob(f"{OUTPUT_CSV}.shard*"))
if shard_files:
    df = pd.concat([pd.read_csv(f) for f in shard_files], ignore_index=True)
    print(f"Loaded {len(shard_files)} shard files, {len(df)} rows total.")
else:
    df = pd.read_csv(OUTPUT_CSV)
    print(f"Loaded {OUTPUT_CSV}, {len(df)} rows.")

# The two arms get well-separated colours (blue vs goldenrod) from
# plot_style, NOT two shades of one hue: they are drawn side by side in every
# epsilon group, so telling them apart at a glance matters more than signalling
# that they are the same underlying mechanism.
ARMS = ["edge_flip_pure_dp", "edge_flip_approx_dp"]
ARM_LABEL = {a: method_label(a) for a in ARMS}
ARM_COLOR = {a: method_color(a) for a in ARMS}

DELTA = sorted(d for d in df["delta"].unique() if d > 0)[0]
Ns = sorted(df["N"].unique())
eps_vals = sorted(df["epsilon"].unique())
true_00 = df["beta_true_00"].iloc[0]
true_01 = df["beta_true_01"].iloc[0]

print(f"delta = {DELTA:g}   |   N in {Ns}   |   {df['rep'].nunique()} reps per cell")
print(f"true beta: intra = {true_00}, inter = {true_01}")

# ── the mechanism-level difference the whole appendix hinges on ────────────
tbl = pd.DataFrame({"epsilon": eps_vals})
tbl["f_pure"] = tbl["epsilon"].map(lambda e: expit(-e))
tbl["f_approx"] = tbl["f_pure"] * (1 - DELTA)
tbl["abs_gap"] = tbl["f_pure"] - tbl["f_approx"]
tbl["rel_gap"] = tbl["abs_gap"] / tbl["f_pure"]
for N in Ns:
    tbl[f"E[diff pairs] N={N}"] = tbl["abs_gap"] * N * (N - 1) / 2
print("\nFlip probability under each formulation (delta enters only as a factor (1 - delta)):")
display(tbl.style.format({"f_pure": "{:.6e}", "f_approx": "{:.6e}", "abs_gap": "{:.3e}",
                          "rel_gap": "{:.1e}",
                          **{f"E[diff pairs] N={N}": "{:.3f}" for N in Ns}}))
print("E[diff pairs] = expected number of node pairs the two formulations would flip\n"
      "differently if run on common random numbers -- i.e. the entire budget of\n"
      "possible disagreement between the arms at this delta.")
df.head()

In [ ]:
# ── Cell 2: THE APPENDIX FIGURE ────────────────────────────────────────────
# Layout: one ROW per N, three COLUMNS (beta intra, beta inter, NMI) -- the
# transpose of the earlier layout, matching the row-per-N figures in the other
# notebooks, with each panel large enough to survive scaling down in LaTeX.
#
# Columns 1-2 are boxplots. Column 3 is NMI as mean +/- 95% bootstrap CI
# rather than a boxplot: NMI here is strongly BIMODAL (VEM-SBM either recovers
# the planted split at ~1.0 or collapses to ~0), so a box spans the whole
# [0, 1] range in every cell and hides exactly the comparison this figure is
# about. The mean with a CI shows whether the two arms sit on top of each
# other, which is the claim being tested; individual reps are drawn as faint
# jittered points so the bimodality stays visible.
#
# Colours/median style come from plot_style.py (shared with every other
# analysis notebook).
# Saved to figures/appendix_edgeflip_puredp_vs_approxdp.{pdf,png}.
import os

import matplotlib.pyplot as plt
import numpy as np
from matplotlib.patches import Patch
from matplotlib.lines import Line2D

from plot_style import (apply_style, figsize, grouped_boxplot, reference_line,
                        BOX_ALPHA, MEDIAN_COLOR, REFERENCE_COLOR)

apply_style()
os.makedirs("figures", exist_ok=True)

plot_df = df.copy()
plot_df["intra_est"] = plot_df[["beta_est_00", "beta_est_11"]].mean(axis=1)

BOX_COLUMNS = [
    ("intra_est",   r"$\hat\beta$ intra-block", true_00),
    ("beta_est_01", r"$\hat\beta$ inter-block", true_01),
]
N_COLS = len(BOX_COLUMNS) + 1

# Per-panel inches before FIG_SCALE -- the single knob for retuning size.
PANEL_W, PANEL_H = 6.0, 4.6

def boot_ci(vals, n_boot=5000, seed=0):
    """Percentile bootstrap CI of the mean -- no normality assumption, which
    matters because these samples are bimodal, not bell-shaped."""
    vals = np.asarray(vals, dtype=float)
    if len(vals) == 0:
        return np.nan, np.nan, np.nan
    if np.ptp(vals) == 0:
        return vals.mean(), vals.mean(), vals.mean()
    rng = np.random.default_rng(seed)
    means = rng.choice(vals, size=(n_boot, len(vals)), replace=True).mean(axis=1)
    return vals.mean(), np.percentile(means, 2.5), np.percentile(means, 97.5)

fig, axes = plt.subplots(
    len(Ns), N_COLS,
    figsize=figsize(PANEL_W * N_COLS, PANEL_H * len(Ns)),
    squeeze=False,
)
width = 0.8 / len(ARMS)

for row, N in enumerate(Ns):
    sub_N = plot_df[plot_df["N"] == N]

    # ── columns 1-2: beta boxplots ────────────────────────────────────────
    for col, (est_col, col_label, ref_val) in enumerate(BOX_COLUMNS):
        ax = axes[row, col]
        grouped_boxplot(ax, sub_N, est_col, methods=ARMS, x_values=eps_vals,
                        x_fmt="{:.3g}")
        reference_line(ax, ref_val)
        ax.set_xlabel(r"$\epsilon$")
        if row == 0:
            ax.set_title(col_label, fontsize=19, pad=12)

    # ── column 3: NMI, mean +/- bootstrap CI ──────────────────────────────
    ax = axes[row, N_COLS - 1]
    rng_jit = np.random.default_rng(0)
    for a_idx, arm in enumerate(ARMS):
        xs, means, los, his = [], [], [], []
        for e_idx, eps in enumerate(eps_vals):
            vals = sub_N[(sub_N["epsilon"] == eps) & (sub_N["method"] == arm)]["nmi"].dropna().values
            if len(vals) == 0:
                continue
            m, lo, hi = boot_ci(vals, seed=e_idx * 10 + a_idx)
            x = e_idx + (a_idx - len(ARMS) / 2 + 0.5) * width
            xs.append(x); means.append(m); los.append(lo); his.append(hi)
            ax.scatter(np.full(len(vals), x) + rng_jit.uniform(-width * 0.25, width * 0.25, len(vals)),
                       vals, s=12, color=ARM_COLOR[arm], alpha=0.30, zorder=2, linewidths=0)
        ax.errorbar(xs, means, yerr=[np.array(means) - np.array(los),
                                     np.array(his) - np.array(means)],
                    fmt="o-", color=ARM_COLOR[arm], markersize=7, linewidth=2.0,
                    capsize=4, zorder=4, label=ARM_LABEL[arm])
    reference_line(ax, 1.0)
    ax.set_ylim(-0.05, 1.12)
    ax.set_xticks(range(len(eps_vals)))
    ax.set_xticklabels([f"{e:.3g}" for e in eps_vals], rotation=45)
    ax.set_xlim(-0.6, len(eps_vals) - 0.4)
    ax.grid(alpha=0.3)
    ax.set_xlabel(r"$\epsilon$")
    if row == 0:
        ax.set_title("NMI (mean $\\pm$ 95% CI)", fontsize=19, pad=12)

    # N labels the row, stated once on the left.
    axes[row, 0].set_ylabel(f"N = {N}\nestimate", fontsize=18)

handles = [Patch(facecolor=ARM_COLOR[a], alpha=BOX_ALPHA, edgecolor="#333333",
                 label=ARM_LABEL[a]) for a in ARMS]
handles += [
    Line2D([0], [0], color=MEDIAN_COLOR, linewidth=2.0, label="median (boxes)"),
    Line2D([0], [0], color=REFERENCE_COLOR, linestyle="--", linewidth=1.6,
           label="true value / NMI = 1"),
]
fig.legend(handles=handles, loc="upper center", ncol=len(handles),
           fontsize=16, frameon=True, bbox_to_anchor=(0.5, 1.0))

fig.suptitle(rf"Edge-flip under Pure DP vs Approximate DP at matched $\epsilon$  "
             rf"($\delta = {DELTA:g}$)", y=1.035)
plt.tight_layout(rect=[0, 0, 1, 0.96])
for ext in ("pdf", "png"):
    fig.savefig(f"figures/appendix_edgeflip_puredp_vs_approxdp.{ext}",
                bbox_inches="tight")
print("saved figures/appendix_edgeflip_puredp_vs_approxdp.{pdf,png}")
plt.show()

In [ ]:
# ── Cell 3: paired per-graph differences (approx - pure) ────────────────────
# Both arms run on the SAME graph within a (N, epsilon, rep), so differencing
# them per rep removes graph variance and isolates the mechanism difference.
# A systematic utility advantage for approximate DP would show up as a
# consistently positive difference; sampling noise shows up as a cloud
# centred on zero.
import numpy as np
import pandas as pd
from scipy import stats

paired_src = df.copy()
paired_src["intra_est"] = paired_src[["beta_est_00", "beta_est_11"]].mean(axis=1)
METRICS = {"nmi": "NMI", "intra_est": r"beta intra", "beta_est_01": r"beta inter"}

wide = paired_src.pivot_table(index=["N", "epsilon", "rep"], columns="dp_type",
                              values=list(METRICS))
rows = []
for metric, label in METRICS.items():
    d = (wide[(metric, "approx")] - wide[(metric, "pure")]).dropna()
    for (N, eps), g in d.groupby(level=["N", "epsilon"]):
        g = g.values
        n = len(g)
        sem = stats.sem(g) if n > 1 and np.ptp(g) > 0 else 0.0
        half = sem * stats.t.ppf(0.975, n - 1) if sem > 0 else 0.0
        if np.ptp(g) > 0:
            _stat, pval = stats.wilcoxon(g)
        else:
            pval = 1.0          # exactly identical in every rep
        rows.append(dict(metric=label, N=N, epsilon=round(eps, 4), n=n,
                         mean_diff=g.mean(), ci_lo=g.mean() - half, ci_hi=g.mean() + half,
                         n_identical=int((g == 0).sum()), wilcoxon_p=pval))

paired = pd.DataFrame(rows)
print("Paired differences, approximate DP minus pure DP (positive = approx better).")
print("n_identical counts reps where the two arms produced bit-identical output.\n")
display(paired.round(4))

# Plot the NMI differences -- the metric the thesis text leads with.
# Layout: one ROW per N, matching the other figures in this notebook.
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
from matplotlib.lines import Line2D

from plot_style import (apply_style, figsize, method_color, MEDIAN_COLOR,
                        REFERENCE_COLOR)

apply_style()

PANEL_W, PANEL_H = 10.0, 4.2

fig, axes = plt.subplots(
    len(Ns), 1,
    figsize=figsize(PANEL_W, PANEL_H * len(Ns)),
    squeeze=False, sharey=True,
)
for row, N in enumerate(Ns):
    ax = axes[row, 0]
    d = (wide[("nmi", "approx")] - wide[("nmi", "pure")]).xs(N, level="N")
    data = [d.xs(e, level="epsilon").values for e in eps_vals]
    bp = ax.boxplot(data, patch_artist=True, widths=0.55,
                    medianprops=dict(color=MEDIAN_COLOR, linewidth=2.0))
    for patch in bp["boxes"]:
        patch.set_facecolor("#BFBFBF")
        patch.set_alpha(0.55)
        patch.set_edgecolor("#333333")
    rng = np.random.default_rng(1)
    for i, vals in enumerate(data, start=1):
        ax.scatter(i + rng.uniform(-0.12, 0.12, len(vals)), vals, s=18,
                   color=method_color("edge_flip_pure_dp"), alpha=0.55,
                   zorder=3, linewidths=0)
    ax.axhline(0.0, color=REFERENCE_COLOR, linestyle="--", linewidth=1.6, alpha=0.8)
    ax.set_xticklabels([f"{e:.3g}" for e in eps_vals], rotation=45)
    ax.set_xlabel(r"$\epsilon$")
    ax.set_ylabel(f"N = {N}\nNMI(approx) $-$ NMI(pure)", fontsize=17)
    ax.grid(alpha=0.3)
    if row == 0:
        ax.set_title("Paired per-graph NMI difference: centred on zero,\n"
                     "no systematic advantage", fontsize=19, pad=12)

handles = [
    Patch(facecolor="#BFBFBF", alpha=0.55, edgecolor="#333333",
          label="per-graph differences"),
    Line2D([0], [0], color=MEDIAN_COLOR, linewidth=2.0, label="median"),
    Line2D([0], [0], color=REFERENCE_COLOR, linestyle="--", linewidth=1.6,
           label="no difference"),
]
fig.legend(handles=handles, loc="upper center", ncol=len(handles),
           fontsize=16, frameon=True, bbox_to_anchor=(0.5, 1.02))

plt.tight_layout(rect=[0, 0, 1, 0.95])
plt.show()

# Power statement: with bimodal NMI and this many reps, say what the data can
# actually exclude rather than claiming equivalence outright.
nmi_diff = (wide[("nmi", "approx")] - wide[("nmi", "pure")]).dropna()
sd = nmi_diff.std()
n_per_cell = paired.query("metric == 'NMI'")["n"].max()
mde = 2.8 * sd / np.sqrt(n_per_cell)
print(f"\nPooled SD of the paired NMI difference: {sd:.3f}")
print(f"With n = {n_per_cell} reps per cell, the smallest per-cell mean difference this\n"
      f"design could detect at 80% power / 5% level is about {mde:.3f} NMI.\n"
      f"So the result reads as 'no difference larger than ~{mde:.2f} NMI', not as proof of\n"
      f"exact equality -- which is the appropriate claim for the appendix.")

In [ ]:
# ── Cell 4: why the two formulations coincide -- delta is the whole story ───
# f_delta / f_pure = (1 - delta) exactly, at every epsilon. So the utility gap
# is controlled by delta alone, and at the thesis's delta = 1e-5 it is a
# relative 1e-5 perturbation of the flip probability. This panel shows where
# delta WOULD start to matter, which is what justifies the qualifier "when
# delta is small" in the text.
import matplotlib.pyplot as plt
import numpy as np
from scipy.special import expit

from plot_style import (apply_style, figsize, method_color, MEDIAN_COLOR,
                        REFERENCE_COLOR)

apply_style()

eps_grid = np.linspace(0.1, 12, 400)
deltas_show = [1e-5, 1e-3, 1e-1, 0.5]
# Gold shades: these curves ARE the approximate-DP mechanism, which is gold
# in every other figure. Pure DP stays black here as the reference curve.
delta_shades = ["#F0D080", "#D4A017", "#A87C10", "#6E5008"]

fig, axes = plt.subplots(1, 2, figsize=figsize(13, 4.8))

ax = axes[0]
ax.plot(eps_grid, expit(-eps_grid), color="black", linewidth=2.4,
        label=r"pure DP:  $f = 1/(1+e^\epsilon)$")
for d, c in zip(deltas_show, delta_shades):
    ax.plot(eps_grid, (1 - d) * expit(-eps_grid), linestyle="--", linewidth=1.8,
            color=c, alpha=0.95, label=rf"approx. DP, $\delta = {d:g}$")
for e in eps_vals:
    if e <= eps_grid.max():
        ax.axvline(e, color="grey", alpha=0.3, linewidth=0.9)
ax.set_xlabel(r"$\epsilon$")
ax.set_ylabel("flip probability")
ax.set_title(r"Flip probability: $f_\delta = (1-\delta)\,f$"
             "\n(small $\\delta$ sits on top of the pure-DP curve)")
ax.grid(alpha=0.3)
ax.legend(fontsize=12)

ax = axes[1]
dd = np.logspace(-6, -0.3, 200)
N_SHADES = ["#B0B0B0", "#707070", "#303030"]   # neutral: these are graph
                                                # sizes, not methods
e_min = min(eps_vals)
for i, N in enumerate(Ns):
    n_pairs = N * (N - 1) / 2
    ax.plot(dd, dd * expit(-e_min) * n_pairs, linewidth=2.2,
            color=N_SHADES[i % len(N_SHADES)], label=f"N = {N}")
ax.axvline(DELTA, color=MEDIAN_COLOR, linestyle="--", linewidth=1.8,
           label=rf"thesis $\delta = {DELTA:g}$")
ax.axhline(1.0, color=REFERENCE_COLOR, linestyle=":", linewidth=1.6, label="one pair")
ax.set_xscale("log")
ax.set_yscale("log")
ax.set_xlabel(r"$\delta$")
ax.set_ylabel("expected # pairs flipped differently")
ax.set_title(rf"Disagreement budget at $\epsilon = {e_min:.2f}$:"
             "\nbelow one pair until $\\delta \\approx 10^{-3}$")
ax.grid(alpha=0.3, which="both")
ax.legend(fontsize=12)

plt.tight_layout()
plt.savefig("figures/appendix_edgeflip_delta_mechanics.pdf", bbox_inches="tight")
plt.savefig("figures/appendix_edgeflip_delta_mechanics.png", bbox_inches="tight")
print("saved figures/appendix_edgeflip_delta_mechanics.{pdf,png}")
plt.show()

# Realized flip counts from the run, as a sanity check on the above.
flips = (df.groupby(["N", "epsilon", "dp_type"])
           .agg(mean_flipped=("n_flipped", "mean"),
                frac_flipped=("frac_flipped", "mean"))
           .unstack("dp_type").round(4))
print("\nRealized flips per arm (mean over reps):")
display(flips)

In [ ]:
# ── Cell 5: summary table for the appendix text ────────────────────────────
import numpy as np
import pandas as pd

s = df.copy()
s["intra_est"] = s[["beta_est_00", "beta_est_11"]].mean(axis=1)
s["abs_err_intra"] = (s["intra_est"] - s["beta_true_00"]).abs()
s["abs_err_inter"] = (s["beta_est_01"] - s["beta_true_01"]).abs()
s["recovered"] = (s["nmi"] > 0.5).astype(float)   # NMI is bimodal; this is the honest summary

out = (s.groupby(["N", "epsilon", "dp_type"])
        .agg(nmi_mean=("nmi", "mean"),
             nmi_median=("nmi", "median"),
             recovery_rate=("recovered", "mean"),
             beta_intra_mean=("intra_est", "mean"),
             abs_err_intra=("abs_err_intra", "mean"),
             beta_inter_mean=("beta_est_01", "mean"),
             abs_err_inter=("abs_err_inter", "mean"),
             mean_flipped=("n_flipped", "mean"),
             n=("nmi", "size"))
        .round(4)
        .reset_index())
display(out)

# One-line verdicts for the appendix paragraph.
piv = out.pivot_table(index=["N", "epsilon"], columns="dp_type",
                      values=["nmi_mean", "recovery_rate", "abs_err_intra", "abs_err_inter"])
gap_nmi = (piv[("nmi_mean", "approx")] - piv[("nmi_mean", "pure")])
gap_rec = (piv[("recovery_rate", "approx")] - piv[("recovery_rate", "pure")])
n_identical_cells = int((gap_nmi == 0).sum())
print(f"\nCells where the two formulations agree EXACTLY: {n_identical_cells} of {len(gap_nmi)} "
      f"(these are the epsilon values at which neither arm flips anything).")
print(f"Mean NMI gap (approx - pure) across cells : {gap_nmi.mean():+.4f}  "
      f"[min {gap_nmi.min():+.4f}, max {gap_nmi.max():+.4f}]")
print(f"Recovery-rate gap across cells            : {gap_rec.mean():+.4f}  "
      f"[min {gap_rec.min():+.4f}, max {gap_rec.max():+.4f}]")
print(f"Sign of the NMI gap: {int((gap_nmi > 0).sum())} cells favour approx, "
      f"{int((gap_nmi < 0).sum())} favour pure, {int((gap_nmi == 0).sum())} tie "
      f"-- i.e. the differences do not point in a consistent direction.")